In [17]:
import datasets

ds = datasets.load_dataset('parquet', data_files='/data/wangsl/datasets/KlearReasoner/train_verl.parquet', split='train')
ds_5k = datasets.load_dataset('parquet', data_files='/data/xieht/deepcoder_processed_5k.parquet', split='train')

In [20]:
import pandas as pd

df = ds.to_pandas()

In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)

sampled = df.sample(n=128, random_state=rng)

In [ ]:
import json
import re

def extract_all_numbers(question):
    # find all numbers include float
    return [s for s in re.findall(r'\d+\.\d+|\d+', question)]

sampled_ds = datasets.Dataset.from_pandas(sampled)

sampled_ds = sampled_ds.filter(lambda x: any([
    ' '.join(extract_all_numbers(test['input'])) in ' '.join(extract_all_numbers(x['extra_info']['question']))
    for test in json.loads(x['extra_info']['tests'])
]))

In [23]:
def input_type(x):
    if isinstance(x, list):
        if len(x) > 0:
            return str(type(list())) + input_type(x[0])
        else:
            return str(type(list()))
    else:
        return str(type(x))

df['extra_info'].groupby(df['extra_info'].apply(lambda x: input_type(json.loads(x['ground_truth'])[0]['input']))).count()

extra_info
<class 'list'><class 'str'>        3
<class 'str'>                  14955
Name: extra_info, dtype: int64

In [24]:
df['extra_info'].groupby(df['extra_info'].apply(lambda x: input_type(json.loads(x['ground_truth'])[0]['output']))).count()

extra_info
<class 'list'><class 'str'>        3
<class 'str'>                  14955
Name: extra_info, dtype: int64